In [ ]:
import torch
import matplotlib.pyplot as plt

**Building the Dataset**

In [ ]:
# load the names dataset from file
from pathlib import Path

data_dir = Path.cwd() / ".." / ".." / "data"


def load_names(path: Path) -> list[str]:
    with path.open("r") as f:
        return f.read().splitlines()


words = load_names(data_dir / "names.txt")
words[:4]

In [ ]:
from makemore.data import make_dataset
from makemore.vocab import Vocab
import string

# build the vocabulary
vocab = Vocab.from_words([string.ascii_lowercase])

# make the dataset
X, Y = make_dataset(words[:5], vocab, block_size=3)

In [ ]:
X.shape, X.dtype, Y.shape, Y.dtype

**Building the Network**

In [ ]:
# controls the size of the embedding
embedding_dimension = 2

# randomly initialize lookup table C
C = torch.randn((len(vocab), embedding_dimension))

In [ ]:
# exploring the equivalence between indexing and matmul
import torch.nn.functional as F

# indexing
print(C[vocab.stoi["e"]])
# matmul
print(
    F.one_hot(torch.tensor(vocab.stoi["e"]), num_classes=len(vocab)).float() @ C
)

In [ ]:
# intuition: embedding all blocks at the same time
print(X[0])
print(C[torch.tensor([0, 0, 0])])
print(C[X].shape)

In [ ]:
emb = C[X]

In [ ]:
# parameters for hidden layer
W1 = torch.randn((6, 100))
b1 = torch.randn(100)

In [ ]:
h = torch.tanh(emb.view((-1, 6)) @ W1 + b1)
h.shape

In [ ]:
# parameters for final layer
W2 = torch.randn((100, len(vocab)))
b2 = torch.randn(len(vocab))

In [ ]:
logits = h @ W2 + b2
logits.shape

In [ ]:
# compute the loss
import torch.nn.functional as F

F.cross_entropy(logits, Y)

**Aside: `F.cross_entropy()`**

In [ ]:
logits = torch.tensor([-100, -103, 0, 100]) - 100
counts = logits.exp()
probs = counts / counts.sum()
probs

**The Complete Forward Pass**

In [ ]:
# embed all blocks for each input example
emb = C[X]
# hidden layer activations
h = torch.tanh(emb.view((-1, 6)) @ W1 + b1)
# logits (activations from output layer)
logits = h @ W2 + b2
# NLL loss
loss = F.cross_entropy(logits, Y)
loss

**Wrapping it in `CharLM` + A Simple Training Loop** 

In [ ]:
from makemore.mlp import MLP

model = MLP(vocab)
logits = model.forward(X)
loss = F.cross_entropy(logits, Y)
loss

In [ ]:
for _ in range(10):
    # forward pass
    logits = model.forward(X)
    loss = F.cross_entropy(logits, Y)
    print(loss.item())

    # backward pass
    for p in model.parameters():
        p.grad = None
    loss.backward()

    # update
    for p in model.parameters():
        p.data += -0.1 * p.grad  # type: ignore

**Computing Hyperparameters**

In [ ]:
hyperparameters = {
    "embedding_dimension": [2, 5, 10],
    "hidden_layer_size": [100, 250, 500],
}

In [ ]:
import itertools
import functools


def hyperparameter_grid(hps: dict[str, list[int]]) -> list[dict[str, int]]:
    inputs = [[{key: v} for v in values] for key, values in hps.items()]
    prod = itertools.product(*inputs)
    return [functools.reduce(lambda a, b: a | b, setting) for setting in prod]

In [ ]:
grid = hyperparameter_grid(hyperparameters)
grid

**A Complete Experiment**

In [ ]:
# perform dataset split
from makemore.data import split

train, dev, test = split(words, fractions=(0.8, 0.1))

# make datasets for each split
X_dev, Y_dev = make_dataset(dev, vocab, block_size=3)
X_test, Y_test = make_dataset(test, vocab, block_size=3)

In [ ]:
import makemore.train as training

train_steps = 200_000
lr_schedule = lambda step: 0.01 if step / train_steps > 0.5 else 0.1

# best setting of hyperparameters
best_setting = {}
# best validation loss
best_valloss = 100.0

for i, setting in enumerate(grid):
    # build the model
    model = MLP(vocab, **setting)
    # build the optimizer
    opt = training.SGD(model.parameters(), lr=0.1)

    # run training
    losses = training.train(
        model, train, opt, steps=50_000, batch_size=64, lr_schedule=lr_schedule
    )

    # compute validation loss
    val_loss = model.loss(X_dev, Y_dev)

    if val_loss < best_valloss:
        best_valloss = val_loss
        best_setting = setting

    print(
        f"[{i+1} / {len(grid)}] evaluated setting: {setting} (n_params={model.num_parameters})"
    )
    print(
        f"[{i+1} / {len(grid)}] final_training_loss={losses[-1]}, val_loss={val_loss.item()}"
    )

In [ ]:
final_model = MLP(vocab, **best_setting)
print(
    f"found best setting: {best_setting} (n_params={final_model.num_parameters})"
)
print(f"best validation loss = {best_valloss}")

In [ ]:
test_loss = final_model.loss(X_test, Y_test)
print(f"test_loss={test_loss}")